# Phase#1 — Mercedes Binary Classifier — Statistics

In [ ]:
import plotly.io as pio

import sys
from pathlib import Path
import joblib, numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "models" / "mercedes_model.pkl").exists() or p.name.startswith("Phase#1"):
            return p
    raise RuntimeError("Cannot locate Phase#1 root. Start Jupyter from Main/ or Main/Phase#1/")

_ROOT         = _find_root()
_MODEL_PATH   = _ROOT / "models" / "mercedes_model.pkl"
_LABELED_PATH = _ROOT / "output" / "1M_labeled_report.csv"
print("Phase#1 root:", _ROOT)

## Model Parameters

In [ ]:
m = joblib.load(_MODEL_PATH)
print(f"Type     : {type(m).__name__}")
print(f"C        : {m.C}  |  solver: {m.solver}  |  features: {m.n_features_in_}  |  iters: {m.n_iter_[0]}")
print(f"Classes  : {list(m.classes_)}  (0=not_mercedes, 1=mercedes)")

coef_df = pd.DataFrame({"feature": m.feature_names_in_, "weight": m.coef_[0]}).sort_values("weight")
fig = px.bar(
    coef_df, x="weight", y="feature", orientation="h",
    title="Feature weights  (+  →  mercedes)",
    color="weight", color_continuous_scale="RdBu", color_continuous_midpoint=0,
)
fig.update_layout(height=500, coloraxis_showscale=False)
fig.show()

## 1M File — Label Distribution

In [ ]:
df = pd.read_csv(_LABELED_PATH, dtype=str)
df["prob_mercedes"] = pd.to_numeric(df["prob_mercedes"], errors="coerce")
total = len(df)

counts = df["decision"].value_counts().reset_index()
counts.columns = ["label", "count"]
counts["pct"] = (counts["count"] / total * 100).round(2)

fig = px.bar(
    counts, x="label", y="count", text="pct",
    title=f"Label distribution — {total:,} articles",
    color="label",
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()
display(counts.to_string(index=False))

## Confidence per Category

In [ ]:
sample = df.sample(min(100_000, len(df)), random_state=42)
fig = px.violin(
    sample, x="decision", y="prob_mercedes",
    title="prob_mercedes distribution by decision",
    box=True, points=False, color="decision",
)
fig.show()

print("\nConfidence stats per decision:")
tbl = df.groupby("decision")["prob_mercedes"].agg(
    mean="mean", p50="median",
    p10=lambda x: x.quantile(0.10),
    p90=lambda x: x.quantile(0.90),
    min="min", max="max",
).round(3)
display(tbl)